# Notebook 09 — The Coding Workflow

**Premise:** The eight preceding notebooks taught the *pieces* — the
loop, the prompts, the skills, the identity, the audit. This
notebook teaches the *workflow* that uses them to build software
the way you'd want a senior engineer at your lab to build it.

It is the same pattern, fractal:

> **Plan · Do · Validate** — at the *workflow* level, and at the
> *skill* level inside every step.

We walk through the seven phases — `steering → brainstorm → build →
deepen → specify → implement → review` — read the actual artifacts
produced at each stage, and unpack *why each phase exists*. The
principled-coder identity governs the entire chain.

By the end you will have:
- Read the principled-coder identity that governs every phase
- Walked through real artifacts from a shipped feature (Kimi provider, SPEC-001)
- Mapped each command to a plan/do/validate role
- Seen the same fractal pattern at workflow level and skill level
- Understood why heavy plan + cheap do works better than the reverse

## Setup

Two small things power this whole notebook: a few `rich` helpers for pretty terminal output, and a pointer to the project's `.claude` directory.

**Imports and the mirror path.** We pull in `rich` for panels and markdown rendering, and point `MIRROR` at the repo's `.claude` directory — every artifact we read lives under it.

In [ ]:
from pathlib import Path
from rich import print
from rich.panel import Panel
from rich.console import Console
from rich.table import Table
from rich.markdown import Markdown

console = Console()
MIRROR = Path('../.claude').resolve()

**One helper: `show()`.** It reads a file under `.claude`, optionally trims to the first `head` lines, and prints it as a bordered, markdown-rendered panel.

In [ ]:
def show(path: str, *, title: str | None = None, head: int | None = None) -> None:
    p = MIRROR / path
    body = p.read_text()
    if head is not None:
        body = '\n'.join(body.splitlines()[:head])
    console.print(Panel(Markdown(body), title=title or path, border_style='cyan'))

## 1. The identity that governs everything

Every command in the chain — `/brainstorm`, `/build`, `/deepen`,
`/specify`, `/implement`, `/review` — references the same identity:
the **principled-coder**. Recall notebook 06: an identity is a
version-controlled file that produces aligned decisions across many
agents and many roles. This is that pattern, applied to coding.

Read the four pillars. Note the explicit ordering: when they
conflict, simplicity wins over cleverness, modularity wins over
monoliths, security wins over convenience, scalability wins over
"fast enough."

**Read the principled-coder identity.** This is the version-controlled file every command in the chain references — its four pillars are the rubric for every phase below.

In [2]:
show('agents/principled-coder.md', title='principled-coder identity', head=70)

╭─────────────────────────────────────────── principled-coder identity ───────────────────────────────────────────╮
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ name: principled-coder version: 1.0.0 last-updated: 2026-04-17 description: Coding identity grounded in four    │
│ non-negotiable priorities — Simplicity, Modularity, Security, Scalability. Every design, build, and planning    │
│ decision is filtered through these pillars in order. Use when writing, reviewing, refactoring, or architecting  │
│ code where these priorities must govern the outcome. tools: Read, Write, Edit, MultiEdit, Bash, Grep, Glob      │
│ model: sonnet                                                                                                   │
│                                                                                                                 │
│ You are a principled coder. Your identity is defined by four priorities that govern every decision — what to    │
│ build, how to build it, and what to refuse. These are not guidelines. They are filters.                         │
│                                                                                                                 │
│ The Four Priorities (Ordered)                                                                                   │
│                                                                                                                 │
│ When priorities conflict, resolve in this order: Simplicity → Modularity → Security → Scalability. Simpler wins │
│ over cleverer. Modular wins over monolithic. Secure wins over convenient. Scalable wins over "fast enough."     │
│                                                                                                                 │
│ 1. Simplicity                                                                                                   │
│                                                                                                                 │
│ Code is read ten times more than written. Optimize for the reader.                                              │
│                                                                                                                 │
│  • Use common, boring patterns over clever ones. If a junior engineer cannot read it cold, rewrite it.          │
│  • Must work out of the box. Zero-config defaults. Sensible fallbacks. No "set up these twelve things first."   │
│  • Avoid premature abstraction. Three similar lines beat a generic helper you might need later.                 │
│  • One function does one thing. If you need "and" to describe it, split it.                                     │
│  • Delete dead code, unused parameters, commented-out blocks. Git remembers so you don't have to.               │
│  • Short files, short functions, flat control flow. Nesting is debt.                                            │
│                                                                                                                 │
│ 2. Modularity                                                                                                   │
│                                                                                                                 │
│ Any section must be replaceable without touching the rest — as long as the input and output contracts hold.     │
│                                                                                                                 │
│  • Enforce hard boundaries between domains: TUI, commands, CLI, teams, agents, run. Logic never bleeds across.  │
│  • Each module owns its state. No shared mutable globals. No reaching into another module's internals.          │
│  • Contracts are explicit: typed inputs, typed outputs

Every phase below applies these pillars *in that order*. A
brainstorm that violates simplicity gets pushed back. A spec that
leaks logic across modules gets rewritten. An implementation
without audit hooks fails review. The identity is the rubric.

## 2. The seven phases

| # | Phase | Role | Output |
|---|---|---|---|
| 0 | `/create-steering-docs` | One-time project context | `product.md`, `tech.md`, `structure.md`, `roadmap.md` |
| 1 | `/brainstorm` | **Plan — WHY** | `.claude/brainstorms/<feature>.md` |
| 2 | `/build` | **Plan — WHAT (decisions)** | entry in `.claude/decisions-log.md` |
| 3 | `/deepen` | **Plan — research enrichment** | enriched build doc |
| 4 | `/specify` | **Plan — HOW (concrete)** | `PRD.md`, `SDD.md`, `PLAN.md` |
| 5 | `/implement` | **Do** | code + tests |
| 6 | `/review` | **Validate** | ADRs + monitoring plan + memory promotion |

Five phases of *plan*, one phase of *do*, one phase of *validate*.
That ratio is **on purpose**. Most teams flip it (one rushed plan
meeting, then five sprints of building, then chaos). The premise
here: 80% of the work happens before any line of production code is
typed.

## Phase 0 — Steering: one-time persistent context

Before any feature, you write four documents that don't change much.
They become the lens through which every future feature gets
evaluated.

- **product.md** — vision, personas, success metrics, business constraints
- **tech.md** — stack, build commands, quality thresholds
- **structure.md** — modules, directories, dependency rules
- **roadmap.md** — phase, themes, what's in scope this quarter

Without steering, every new feature re-litigates the same five
questions ("who's this for?" "what stack?" "what counts as done?").
*With* steering, the agent can read the docs once and stop asking.

**A real steering doc.** Here's the top of an actual `product.md` — the kind of stable context the agent reads once instead of re-asking every feature.

In [3]:
show('examples/steering/product.md', title='Real example: arc/.claude/steering/product.md', head=40)

╭───────────────────────────────── Real example: arc/.claude/steering/product.md ─────────────────────────────────╮
│                                              Product Context — Arc                                              │
│                                                                                                                 │
│ ▌ Stable product context that informs all feature specifications. Feature-specific details go in                │
│ ▌ .claude/specs/{feature}/.                                                                                     │
│                                                                                                                 │
│ Validation Checklist                                                                                            │
│                                                                                                                 │
│  • [x] Vision statement defined                                                                                 │
│  • [x] Personas documented                                                                                      │
│  • [x] User journey framework established                                                                       │
│  • [x] Success metrics framework defined (security/compliance variant — not SaaS)                               │
│  • [x] Business constraints listed                                                                              │
│  • [x] Domain glossary populated                                                                                │
│  • [x] No [NEEDS CLARIFICATION] markers                                                                         │
│                                                                                                                 │
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ Vision & Mission                                                                                                │
│                                                                                                                 │
│ Vision                                                                                                          │
│                                                                                                                 │
│ A world where every autonomous agent action — every prompt, tool call, byte read or written — is attributable,  │
│ authorized, and audit-evident, so AI can be deployed in environments that today refuse to run it.               │
│                                                                                                                 │
│ Mission                                                                                                         │
│                                                                                                                 │
│ Arc is a security-first agentic framework that makes the defensible deployment of AI agents the default, not    │
│ the upgrade path. Every LLM call, every tool dispatch, every artifact load passes through Identity → Sign →     │
│ Authorize → Audit. These four pillars are universal across all tiers; what varies is stringency, not            │
│ enforcement.                                                                                                    │
│                                                                                                                 │
│ Value Proposition                                                                                               │
│                                                                                                                 │
│ Most agent frameworks optimize for "look how fast I ca

## Phase 1 — `/brainstorm`: WHY (plan, layer 1)

Brainstorm explores the *reason for building*. It does NOT make
design decisions — that's `/build`. The output is a structured
exploration: inspiration, audience, use cases, desired outcomes,
guiding principles.

**Why first?** If you skip this, you build a technically correct
thing nobody wanted. The cheapest mistake to fix is the one caught
before any architecture is drawn.

**The `/brainstorm` command.** First the command file itself — the instructions that drive the WHY-exploration dialogue.

In [ ]:
show('commands/brainstorm.md', title='/brainstorm command', head=30)

**A real brainstorm artifact.** And here's what one produces: a structured exploration of inspiration, audience, use cases, and principles.

In [ ]:
show('examples/brainstorms/2026-04-27-kimi-provider.md', title='Real brainstorm: Kimi provider', head=40)

**Plan/Do/Validate inside `/brainstorm`:**
- *Plan*: assess clarity — is brainstorming even needed?
- *Do*: collaborative dialogue, one question at a time (inspiration → audience → use cases → outcomes → principles)
- *Validate*: scope check — what's IN, what's OUT, what stays for later

## Phase 2 — `/build`: WHAT (plan, layer 2 — decisions)

`/build` walks through every design decision a feature needs,
**one at a time**, with tradeoffs. Each option is ranked by the
principled-coder pillars (Simplicity → Modularity → Security →
Scalability). When a federal mandate (NIST, FedRAMP, CMMC) dictates
exactly one answer, the build skill *auto-applies* the decision and
logs the citation rather than asking.

The output: an entry in `decisions-log.md` — a flat table of every
decision, the chosen option, and the tier-specific notes. This is
the artifact the spec consumes next.

**The `/build` command.** It walks every design decision one at a time, ranking options by the principled-coder pillars.

In [5]:
show('commands/build.md', title='/build command', head=40)

╭──────────────────────────────────────────────── /build command ─────────────────────────────────────────────────╮
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ description: "Interactive decision walker — walks through every design decision with tradeoffs, logs choices,   │
│ produces decision document for /specify. Ranks all outputs by simplicity → security → scalability →             │
│ compliance." argument-hint: "[feature name, brainstorm file path, or 'resume' to continue]" allowed-tools:      │
│ ["Read", "Grep", "Glob", "AskUserQuestion", "Write"]                                                            │
│                                                                                                                 │
│                                   /build - Interactive Design Decision Walker                                   │
│                                                                                                                 │
│ Feature: $ARGUMENTS                                                                                             │
│                                                                                                                 │
│ Coding Identity                                                                                                 │
│                                                                                                                 │
│ Apply the principled-coder framework (@agents/principled-coder.md) to every decision, tradeoff, and             │
│ recommendation. The four pillars govern in this order:                                                          │
│                                                                                                                 │
│  1 Simplicity — boring patterns, works out of the box, readable cold                                            │
│  2 Modularity — hard boundaries across TUI, commands, CLI, teams, agents, run; no logic bleed                   │
│  3 Security — open-by-default or locked-by-default controlled by config; fed on-prem capable (DOE, National     │
│    Labs, NASA)                                                                                                  │
│  4 Scalability — stateless, horizontal; 10k+ tx/min across 10–20 coordinated instances                          │
│                                                                                                                 │
│ When priorities conflict, resolve in this order.                                                                │
│                                                                                                                 │
│ What This Does                                                                                                  │
│                                                                                                                 │
│ Walks through every design decision a feature requires — interactively, one at a time. Presents tradeoffs, asks │
│ for your call, logs decisions with reasoning. Produces a decision document that /specify and /deepen consume.   │
│                                                                                                                 │
│ Inspired by the arcllm builder pattern: Concept → Decision → Log → Next.                                        │
│                                                                                                                 │
│ Priority Framework                                                                                              │
│                                                                                                                 │
│ All decisions are evaluated and ranked by the principl

**Plan/Do/Validate inside `/build`:**
- *Plan*: present the decision + ranked options with tradeoffs
- *Do*: ask the user (or auto-apply if federally mandated)
- *Validate*: log decision with reasoning + tier impact, repeat for next decision

## Phase 3 — `/deepen`: research enrichment (plan, layer 3)

Now that decisions are made, `/deepen` spawns parallel research
agents to enrich each decision with best practices, edge cases,
past solutions, and external documentation. This is where the 80/20
split becomes real — heavy research **before** any code, in
parallel, in minutes.

Critically, `/deepen` searches `.claude/solutions/` — your
team's archive of previously-solved problems. Solve once, apply
forever.

**The `/deepen` command.** It spawns parallel research agents and searches the solutions archive to enrich each decision before any code is written.

In [6]:
show('commands/deepen.md', title='/deepen command', head=30)

╭──────────────────────────────────────────────── /deepen command ────────────────────────────────────────────────╮
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ description: "Enrich build decisions or plans with parallel research — best practices, edge cases, solutions    │
│ archive, external docs" argument-hint: "[feature name, build path, or spec ID]" allowed-tools: ["Task", "Read", │
│ "Grep", "Glob", "Write"]                                                                                        │
│                                                                                                                 │
│                                          /deepen - Research Enrichment                                          │
│                                                                                                                 │
│ Target: $ARGUMENTS                                                                                              │
│                                                                                                                 │
│ Coding Identity                                                                                                 │
│                                                                                                                 │
│ Apply the principled-coder framework (@agents/principled-coder.md) to all research, recommendations, and        │
│ enrichment. Every finding must be filtered through the four pillars in order:                                   │
│                                                                                                                 │
│  1 Simplicity — boring patterns, works out of the box, readable cold                                            │
│  2 Modularity — hard boundaries across TUI, commands, CLI, teams, agents, run; no logic bleed                   │
│  3 Security — open-by-default or locked-by-default via config; fed on-prem capable (DOE, National Labs, NASA)   │
│  4 Scalability — stateless, horizontal; 10k+ tx/min across 10–20 coordinated instances                          │
│                                                                                                                 │
│ Research outputs must explicitly call out: scalability ceiling, on-prem/air-gapped posture, and module-boundary │
│ implications. When priorities conflict, resolve in this order.                                                  │
│                                                                                                                 │
│ What This Does                                                                                                  │
│                                                                                                                 │
│ Spawns parallel research agents to enrich a build decision document or PLAN with best practices, edge cases,    │
│ past solutions, and external documentation. Makes the 80/20 split real — heavy research BEFORE any code.        │
│                                                                                                                 │
│ How It Works                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  Load target (build decisions or PLAN.md)                                                                       │
│     ↓                                                                                                           │
│                                                       

**Plan/Do/Validate inside `/deepen`:**
- *Plan*: parse the build doc into sections, match each to a research agent
- *Do*: spawn parallel agents — best-practices researcher, edge-case finder, solutions-archive searcher, doc-fetcher
- *Validate*: synthesize into the build doc, flagging conflicts between sources

## Phase 4 — `/specify`: HOW (plan, layer 4 — concrete tasks)

`/specify` converts decisions + research into three artifacts:

- **PRD.md** — Product Requirements: what the feature does, for whom, success criteria, what's out of scope.
- **SDD.md** — Solution Design: architecture, components, contracts, module boundaries.
- **PLAN.md** — Implementation Plan: ordered tasks with checkboxes, file locations, acceptance per task.

The three docs are linked by **traceability**: every PRD requirement
maps to an SDD component maps to a PLAN task. If a task can't be
traced back to a requirement, it doesn't exist.

**The PRD — what and for whom.** Product Requirements: the feature's purpose, success criteria, and explicit out-of-scope.

In [ ]:
show('examples/spec-kimi/PRD.md', title='Real PRD: Kimi provider', head=30)

**The SDD — how, structurally.** Solution Design: architecture, components, contracts, and module boundaries, each tracing back to a PRD requirement.

In [ ]:
show('examples/spec-kimi/SDD.md', title='Real SDD: Kimi provider', head=30)

**The PLAN — ordered, traceable tasks.** Implementation Plan: checkboxed tasks with file locations, each mapping to an SDD component.

In [ ]:
show('examples/spec-kimi/PLAN.md', title='Real PLAN: Kimi provider', head=30)

Note the discipline: PRD lists requirements with IDs (FR-1, FR-2,
NFR-1...). SDD references those IDs. PLAN references the SDD
components. The chain is auditable end-to-end.

**Plan/Do/Validate inside `/specify`:**
- *Plan*: validate intake (clarity, completeness), detect feature type (API/UI/DB/Integration)
- *Do*: generate PRD → SDD → PLAN, each consuming the previous
- *Validate*: 3-Cs check — Completeness, Consistency, Correctness — and confidence-based routing (fast-track simple features, phase-gates for complex)

## Phase 5 — `/implement`: Do

Now — and *only* now — code gets written. `/implement` reads the
PLAN, categorizes tasks by specialist (test-impl, backend, db, etc.),
spawns parallel agent swarms, and runs **TDD per task**.

**TDD is the Plan/Do/Validate inside `/implement`:**
- *Plan*: write the failing test that captures the desired behavior
- *Do*: write the minimum code to pass it
- *Validate*: run the test (and the rest of the suite); if it fails, fix the *implementation*, not the test

The implement command also applies the four pillars *per task*:
every spawned specialist inherits the principled-coder identity
and refuses code that violates simplicity / modularity / security /
scalability — even if the test passes.

**The `/implement` command.** Only now does code get written — TDD per task, with every spawned specialist inheriting the principled-coder identity.

In [8]:
show('commands/implement.md', title='/implement command', head=40)

╭────────────────────────────────────────────── /implement command ───────────────────────────────────────────────╮
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ description: "Execute implementation with parallel specialist agent swarms. Spawns appropriate agents based on  │
│ task requirements for maximum velocity." argument-hint: "spec-id [phase] (e.g., '001' or '001 3' or             │
│ 'user-auth')" allowed-tools: ["Task", "Read", "Write", "Edit", "Bash", "Grep", "Glob", "TaskCreate",            │
│ "TaskUpdate", "AskUserQuestion"] version: 3.1.0 last-updated: 2026-04-18                                        │
│                                                                                                                 │
│ ▌ Sub-agent policy: Sub-agents ARE allowed during /implement (workflow-enforcement.md updated 2026-04-18).      │
│ ▌ Spawn specialist swarms for parallel work; verify outputs by reading changed files; do NOT trust agent        │
│ ▌ self-reports as completion proof.                                                                             │
│                                                                                                                 │
│                                     /implement - Swarm-Based Implementation                                     │
│                                                                                                                 │
│ Specification: $ARGUMENTS                                                                                       │
│                                                                                                                 │
│ Coding Identity                                                                                                 │
│                                                                                                                 │
│ Apply the principled-coder framework (@agents/principled-coder.md) to every line of code written. Specialist    │
│ agents spawned by this command inherit these four pillars in order:                                             │
│                                                                                                                 │
│  1 Simplicity — boring patterns, works out of the box, readable cold; no premature abstraction                  │
│  2 Modularity — respect domain boundaries (TUI, commands, CLI, teams, agents, run); a single task must not      │
│    cross them                                                                                                   │
│  3 Security — no secrets in code or logs; validate at trust boundaries; config-gated open vs locked posture     │
│  4 Scalability — stateless handlers, idempotent operations, bounded queues; concurrency-safe primitives only    │
│                                                                                                                 │
│ TDD is mandatory: test first, verify it fails for the right reason, minimal code to pass, refactor. When        │
│ priorities conflict, resolve in this order.                                                                     │
│                                                                                                                 │
│ Overview                                                                                                        │
│                                                                                                                 │
│ Spawns parallel specialist agent swarms to execute implementation plans. Analyzes PLAN tasks and delegates to   │
│ appropriate specialists simultaneously.                                                                         │
│                                                       

## Phase 6 — `/review`: Validate

Implementation that passes its own tests has not been reviewed.
`/review` spawns a **separate** swarm with no investment in the
code that was just written:

- security-engineer (OWASP, secrets, trust boundaries)
- architect-reviewer (boundaries, SDD compliance)
- code-reviewer (DRY, SOLID, readability)
- code-simplifier (can this be smaller?)
- coverage-analyzer (test gaps by business impact)
- performance-engineer (N+1, memory, hot paths)

A PR fails review if it violates *any* pillar — even if the tests
pass. Review then produces:
- **ADRs** — architecture decision records, signed and dated, captured for posterity
- **Monitoring plan** — what to watch in prod after deploy
- **Memory promotion** — extracts globally-useful learnings from the spec README into long-term memory

**A real ADR from `/review`.** Review produces architecture decision records — signed, dated, and version-controlled for posterity.

In [9]:
show('examples/adrs/ADR-017A-opt-in-policy-pipeline.md', title='Real ADR produced by /review', head=40)

╭───────────────────────────────────────── Real ADR produced by /review ──────────────────────────────────────────╮
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│ id: ADR-017A title: Opt-in Policy Pipeline at Registry Construction status: accepted date: 2026-04-18 spec:     │
│ SPEC-017 tags:                                                                                                  │
│                                                                                                                 │
│  • policy                                                                                                       │
│  • tool-registry                                                                                                │
│  • backward-compatibility                                                                                       │
│  • security                                                                                                     │
│                                                                                                                 │
│ --------------------------------------------------------------------------------------------------------------- │
│                                                                                                                 │
│                            ADR-017A: Opt-in Policy Pipeline at Registry Construction                            │
│                                                                                                                 │
│ Status                                                                                                          │
│                                                                                                                 │
│ Accepted (2026-04-18). Implemented in arcagent/core/tool_registry.py.                                           │
│                                                                                                                 │
│ Context                                                                                                         │
│                                                                                                                 │
│ SPEC-017 specifies the 5-layer tool policy pipeline (Global → Provider → Agent → Team → Sandbox) as the         │
│ authoritative deny path — every tool call MUST flow through it (R-010, R-011).                                  │
│                                                                                                                 │
│ The existing ToolRegistry has >500 dependent unit and integration tests. Many of them construct a registry with │
│ no policy context whatsoever (fixtures for isolated tool tests). A full cut-over to mandatory pipeline would    │
│ break ~100 tests and force every caller to construct a full pipeline just to register a tool — a significant    │
│ regression in developer ergonomics for test fixtures.                                                           │
│                                                                                                                 │
│ Decision                                                                                                        │
│                                                                                                                 │
│ ToolRegistry(policy_pipeline=None) is an optional constructor argument.                                         │
│                                                                                                                 │
│  • When policy_pipeline is provided, every dispatch goes through it first. PolicyDenied is raised on deny; the  │
│    tool's execute() is never reached. Fail-closed on l

**Plan/Do/Validate inside `/review`:**
- *Plan*: load spec + diff, search solutions archive for relevant prior reviews
- *Do*: spawn parallel reviewers, each filtering through one pillar
- *Validate*: synthesize findings, generate ADRs for accepted-as-is divergences from spec, write monitoring plan

## 3. The fractal — same pattern, two scales

| Scale | Plan | Do | Validate |
|---|---|---|---|
| **Workflow** (whole feature) | brainstorm + build + deepen + specify | implement | review |
| **Skill** (single phase) | gather inputs, list options | act on chosen option | check the action against the input contract |
| **TDD** (single task) | write failing test | write minimum code | run test, run suite, run lint |
| **Decision** (single tradeoff) | enumerate options ranked by pillars | user picks (or federal auto-apply) | log to decisions-log with reasoning |

The same shape, all the way down. **Plan-Do-Validate is fractal.**
Once you see it once, you see it everywhere — in the loop (notebook
03), in prompt design (notebook 02), in identity-driven decisions
(notebook 06), and in the audit chain (notebook 07) which validates
the whole sequence after the fact.

## 4. Why this works

1. **Cheap iteration before expensive iteration.** Throwing away a
   brainstorm costs 10 minutes. Throwing away a sprint of code costs
   2 weeks. The chain front-loads the cheap mistakes.
2. **Explicit identity, not implicit taste.** The principled-coder
   pillars are written down. Anyone (or any agent) can apply them.
   No "vibes-based" decisions.
3. **Validation by a different agent.** `/review` is not the agent
   that did `/implement`. The validator has no investment in the
   work — it's harder for it to rationalize a problem.
4. **Every phase produces a markdown artifact.** Brainstorms,
   decisions, PRDs, SDDs, PLANs, ADRs — all version-controlled,
   diffable, reviewable, blamable. "Why did we choose X?" has an
   answer with a date and a SHA.
5. **Federal auto-apply.** Rules that are mandated (NIST 800-53
   AU-2, etc.) don't get asked — they get applied with the citation.
   Saves time, prevents drift.
6. **Solutions archive compounds.** Every `/review` can promote a
   learning to `.claude/solutions/{category}/`. Next `/brainstorm`
   on a similar topic surfaces the prior solution. The team gets
   smarter with every feature.

## 5. Browse any command — interactive picker

Every command in `.claude/commands/` is plain markdown. Pick one
from the dropdown to read it. *Don't* skip this — most of the
subtlety in the workflow lives in the command files.

**Build the command picker.** List every command file and create a dropdown plus an output area to render the selected one.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

cmd_files = sorted(p.name for p in (MIRROR / 'commands').glob('*.md'))
cmd_dd = widgets.Dropdown(options=cmd_files, value='brainstorm.md', description='Command:', style={'description_width': 'initial'})
view = widgets.Output()

**Wire it up and display.** The callback re-renders the chosen command on every change; we trigger it once and show the widget.

In [ ]:
def _show(_):
    view.clear_output()
    with view:
        show(f'commands/{cmd_dd.value}', title=cmd_dd.value, head=80)

cmd_dd.observe(_show, names='value')
_show(None)
display(widgets.VBox([cmd_dd, view]))

## 6. Try it on a tiny feature — fill in your own

The methodology compresses well — for a small feature you can
run all seven phases in under an hour. The workflow is most
instructive when the feature is *yours*.

✅ **TODO:** Pick a real, small feature you'd build at your lab.
Examples that work well:

- *"Add a `--quiet` flag to our run wrapper that suppresses
  streaming output."*
- *"Emit a warning when 80% of monthly storage budget is used."*
- *"Auto-tag any incident that mentions OST eviction with
  `class:storage-fault`."*
- *"Add a daily report of jobs that ran > 24h on a single node."*

Fill the cells below with your version. Don't run all seven
phases in the notebook — sketch each in 5–10 lines. The point is
to *feel* the ratio: 80% planning artifacts, 20% code.

**Step 1 — name your feature.** One sentence. Pick something small and real you'd actually build at your lab.

In [ ]:
# ✅ TODO — your feature in one sentence.
MY_FEATURE = 'TODO — name your feature here'

**Step 2 — sketch the brainstorm (WHY).** Three to five lines: why it matters, who benefits, the measurable outcome, and what's out of scope.

In [ ]:
# ✅ TODO — your brainstorm (3-5 lines):
MY_BRAINSTORM = '''
Why: TODO — what changes for the user when this ships?
Audience: TODO — who is the primary beneficiary?
Outcomes: TODO — measurable. Within scope: TODO. Out of scope: TODO.
'''

**Step 3 — sketch the decisions (WHAT).** A handful of design tradeoffs with the option you'd pick and why.

In [ ]:
# ✅ TODO — your decisions table (5-8 rows):
MY_DECISIONS = '''
| Decision | Options | Chosen | Why |
|---|---|---|---|
| Where does the flag live? | CLI / config / env | TODO | TODO |
| What's the default? | TODO | TODO | TODO |
| Backwards compat? | strict / lax | TODO | TODO |
'''

**Step 4 — sketch the deepen + specify layers.** Three things research would dig into, the PRD requirement IDs, and the first traceable plan task.

In [ ]:
# ✅ TODO — three questions /deepen would research in parallel:
MY_DEEPEN_QUESTIONS = ['TODO', 'TODO', 'TODO']

# ✅ TODO — sketch PRD requirement IDs (FR-1, NFR-1, ...):
MY_PRD_REQUIREMENTS = ['FR-1: TODO', 'NFR-1: TODO']

# ✅ TODO — first PLAN task (ordered, traceable to PRD):
MY_FIRST_TASK = 'TASK-001: TODO (covers FR-1)'

**Step 5 — render your sketch.** Print the planning artifacts side by side and feel the ratio: lots of planning, very little code.

In [ ]:
console.print(Panel(MY_FEATURE, title='Feature', border_style='cyan'))
console.print(Panel(MY_BRAINSTORM, title='Brainstorm sketch', border_style='yellow'))
console.print(Panel(MY_DECISIONS, title='Decisions sketch', border_style='green'))
console.print(f'\nDeepen research questions: {MY_DEEPEN_QUESTIONS}')
console.print(f'PRD reqs: {MY_PRD_REQUIREMENTS}')
console.print(f'First plan task: {MY_FIRST_TASK}')

## Takeaway

- The workflow is **identity-governed plan/do/validate, fractal**.
- 5 phases of plan, 1 of do, 1 of validate. The ratio is the
  point.
- Every phase outputs a markdown artifact in version control.
  The history of the *thinking* is preserved alongside the
  history of the *code*.
- Skills, identity, audit, and the loop (notebooks 02–08) are
  the ingredients. **This notebook is the recipe.**

**Next:** [10 — Your Workflow](./10_your_workflow.ipynb). Apply
the recipe to a real lab problem.